### 환경설정

In [1]:
import JAEN
from JAEN import list_data, download_data, Project

In [5]:
project_name = "타이타닉 생존자 예측" # 프로젝트 이름(수정하지 마세요)
email = "scsa2406" # 이메일(본인 SCSA 이메일 아이디 입력)

class_info = {
    'edu_name':'SCSA', # 과정명
    'edu_rnd':'24기', # 차수
    'edu_class':'SW' # "없음"이면 빈 문자열
}

# 프로젝트 생성
pjt = Project(project_name=project_name, class_info=class_info, email=email)

### 모듈 import 

필요한 모듈을 import 합니다. 필요에 따라 추가 패키지를 load 할 수 있습니다.

In [58]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os

# 경고 무시
warnings.filterwarnings('ignore')

## 다운로드 및 데이터 로드

In [151]:
download_data(project_name)

파일 다운로드 완료

데이터셋: 타이타닉 생존자 예측
파일경로: data\submission.csv

파일 다운로드 완료

데이터셋: 타이타닉 생존자 예측
파일경로: data\test.csv

파일 다운로드 완료

데이터셋: 타이타닉 생존자 예측
파일경로: data\train.csv



In [152]:
# 데이터 로드
train = pd.read_csv("data/train.csv")
test = pd.read_csv("data/test.csv")
submission = pd.read_csv("data/submission.csv")
submission # 제출 파일

,PassengerId,Survived
0,892,NaN
1,893,NaN
2,894,NaN
3,895,NaN
4,896,NaN
...,...,...
413,1305,NaN
414,1306,NaN
415,1307,NaN
416,1308,NaN


In [153]:
train.head()
test

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,892,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,NaN,Q
1,893,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,NaN,S
2,894,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,NaN,Q
3,895,3,"Wirz, Mr. Albert",male,27.0,0,0,315154,8.6625,NaN,S
4,896,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",female,22.0,1,1,3101298,12.2875,NaN,S
...,...,...,...,...,...,...,...,...,...,...,...
413,1305,3,"Spector, Mr. Woolf",male,NaN,0,0,A.5. 3236,8.0500,NaN,S
414,1306,1,"Oliva y Ocana, Dona. Fermina",female,39.0,0,0,PC 17758,108.9000,C105,C
415,1307,3,"Saether, Mr. Simon Sivertsen",male,38.5,0,0,SOTON/O.Q. 3101262,7.2500,NaN,S
416,1308,3,"Ware, Mr. Frederick",male,NaN,0,0,359309,8.0500,NaN,S


## ↓↓↓ 코드 구현 ↓↓↓
### 여기서부터 코드를 작성/수정하세요
#### - 데이터셋 로드, 전처리(필요시), 모델생성/컴파일, 학습

In [154]:
train.isnull().sum()
test.isnull().sum()

PassengerId      0
Pclass           0
Name             0
Sex              0
Age             86
SibSp            0
Parch            0
Ticket           0
Fare             1
Cabin          327
Embarked         0
dtype: int64

In [155]:
train['Age'].fillna(train['Age'].mode()[0], inplace = True)
test['Age'].fillna(train['Age'].mode()[0], inplace = True)

In [156]:
train['Sex'] = train['Sex'].apply(lambda x: 0 if x=='male' else 1)
test['Sex'] = test['Sex'].apply(lambda x: 0 if x=='male' else 1)

In [85]:
train.corr()

,PassengerId,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked,family
PassengerId,1.000000,-0.005007,-0.035144,-0.042939,0.036186,-0.057527,-0.001652,0.012658,-0.030467,-0.040143
Survived,-0.005007,1.000000,-0.338481,0.543351,-0.052872,-0.035322,0.081629,0.257307,0.106811,0.016639
Pclass,-0.035144,-0.338481,1.000000,-0.131900,-0.356187,0.083081,0.018443,-0.549500,0.045702,0.065997
Sex,-0.042939,0.543351,-0.131900,1.000000,-0.073377,0.114631,0.245489,0.182333,0.116569,0.200988
Age,0.036186,-0.052872,-0.356187,-0.073377,1.000000,-0.232411,-0.155118,0.107554,-0.047625,-0.236339
SibSp,-0.057527,-0.035322,0.083081,0.114631,-0.232411,1.000000,0.414838,0.159651,-0.059961,0.890712
Parch,-0.001652,0.081629,0.018443,0.245489,-0.155118,0.414838,1.000000,0.216225,-0.078665,0.783111
Fare,0.012658,0.257307,-0.549500,0.182333,0.107554,0.159651,0.216225,1.000000,0.062142,0.217138
Embarked,-0.030467,0.106811,0.045702,0.116569,-0.047625,-0.059961,-0.078665,0.062142,1.000000,-0.080281
family,-0.040143,0.016639,0.065997,0.200988,-0.236339,0.890712,0.783111,0.217138,-0.080281,1.000000


In [157]:
train['Embarked'].value_counts()
train['Embarked'] = train['Embarked'].apply(lambda x: 2 if x=='Q' else 1 if x=='C' else 0)
test['Embarked'] = test['Embarked'].apply(lambda x: 2 if x=='Q' else 1 if x=='C' else 0)

In [158]:
train['family']=train['SibSp']+train['Parch']
test['family']=test['SibSp']+test['Parch']

In [159]:
test['Fare'].fillna(test['Fare'].mean(), inplace = True)

In [160]:
X_train = train[['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked', 'family']]
Y_train = train.iloc[:,1]
X_test = test[['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked', 'family']]

In [169]:
from sklearn.model_selection import train_test_split 
x_train, x_test, y_train, y_test = train_test_split(X_train, Y_train, stratify=Y_train, random_state=0)

In [162]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, StackingClassifier

knn2 = KNeighborsClassifier(n_neighbors=2)
knn3 = KNeighborsClassifier(n_neighbors=3)
knn4 = KNeighborsClassifier(n_neighbors=4)
knn5 = KNeighborsClassifier(n_neighbors=5)
lr = LogisticRegression(max_iter=10000)

dt3 = DecisionTreeClassifier(max_depth=3)
dt5 = DecisionTreeClassifier(max_depth=5)
dt10 = DecisionTreeClassifier(max_depth=10)

rand5 = RandomForestClassifier(max_depth=5)
rand = RandomForestClassifier()
gradient = GradientBoostingClassifier()

estimators = [('rf', RandomForestClassifier()),
               ('gb', GradientBoostingClassifier())]
stack = StackingClassifier(estimators=estimators,
                          final_estimator=LogisticRegression())

In [163]:
for idx, model in enumerate([knn2, knn3, knn4, knn5, lr, dt3, dt5, dt10, rand5, rand, gradient, stack]):
    model.fit(X_train, Y_train)

In [135]:
names = ['knn2', 'knn3', 'knn4', 'knn5', 'lr', 'dt3', 'dt5', 'dt10', 'rand5', 'rand', 'gradient', 'stack']
for idx, model in enumerate([knn2, knn3, knn4, knn5, lr, dt3, dt5, dt10, rand5, rand, gradient, stack]):
    model.fit(x_train, y_train)
    name = names[idx]
    train_score = model.score(x_train, y_train)*100
    test_score = model.score(x_test, y_test)*100
    print(f'{name} Train Accuracy: {train_score:.2f}%')
    print(f'{name} Test Accuracy: {test_score:.2f}%')
    print()

knn2 Train Accuracy: 84.73%
knn2 Test Accuracy: 69.96%

knn3 Train Accuracy: 83.98%
knn3 Test Accuracy: 70.85%

knn4 Train Accuracy: 80.24%
knn4 Test Accuracy: 69.96%

knn5 Train Accuracy: 78.89%
knn5 Test Accuracy: 70.85%

lr Train Accuracy: 80.24%
lr Test Accuracy: 78.48%

dt3 Train Accuracy: 84.28%
dt3 Test Accuracy: 78.92%

dt5 Train Accuracy: 85.48%
dt5 Test Accuracy: 82.51%

dt10 Train Accuracy: 92.51%
dt10 Test Accuracy: 81.17%

rand5 Train Accuracy: 85.33%
rand5 Test Accuracy: 82.96%

rand Train Accuracy: 98.65%
rand Test Accuracy: 80.72%

gradient Train Accuracy: 91.32%
gradient Test Accuracy: 82.96%

stack Train Accuracy: 94.16%
stack Test Accuracy: 82.96%



In [136]:
from sklearn.ensemble import VotingClassifier
hard = VotingClassifier([('dt5', dt5), ('dt10', dt10), ('rand', rand), ('gradient', gradient), ('stack', stack)])
soft = VotingClassifier([('dt5', dt5), ('dt10', dt10), ('rand', rand), ('gradient', gradient), ('stack', stack)])

names = ['hard', 'soft']
for idx, model in enumerate([hard, soft]):
    model.fit(x_train, y_train)
    name = names[idx]
    train_score = model.score(x_train, y_train)*100
    test_score = model.score(x_test, y_test)*100
    print(f'{name} Train Accuracy: {train_score:.2f}%')
    print(f'{name} Test Accuracy: {test_score:.2f}%')
    print()

hard Train Accuracy: 93.71%
hard Test Accuracy: 82.96%

soft Train Accuracy: 93.71%
soft Test Accuracy: 82.96%



In [188]:
# 아래의 코드는 필요시 수정해서 사용합니다
from sklearn.preprocessing import StandardScaler, MinMaxScaler
X_scaled_train = MinMaxScaler().fit_transform(X_train)
X_scaled_test =  MinMaxScaler().fit_transform(X_test)

gradient = GradientBoostingClassifier().fit(X_scaled_train, Y_train)
pred = gradient.predict(X_scaled_test)

your_answer = pred
your_answer

submission['Survived'] = your_answer
pjt.submit(submission)

파일을 저장하였습니다. 파일명: submission-12-20-12.csv
제출 여부 :success
오늘 제출 횟수 : 16
제출 결과:0.7822966507177034


In [165]:
submission['Survived'] = your_answer

### 제출 횟수 제한 있습니다. (총 30회)

In [166]:
pjt.submit(submission)

파일을 저장하였습니다. 파일명: submission-12-10-00.csv
제출 여부 :success
오늘 제출 횟수 : 2
제출 결과:0.7464114832535885
